# Chat With Documents Using Hugging Face Open-Source LLMs

## Overview of the Notebook
- So far we explored closed-source projects or paid models and Ollama hosted open source models.
- Now lets explore the world of open-source models available on Hugging Face.
- Build a question answering system where users can upload PDFs and chat with their content.

## Learning Objectives
- Learn what Hugging-Face is by exploring the Hub and Understanding its role in the AI ecosystem.
- Use the Transformers library to work with Hugging-Face models in Python.
- Load Opne-Source models by downloading pre-trained model weights and tokenizers from Hugging-Face.
- Run models efficiently using quantization techniques like bitsandbytes to fit them within GPU memory.
- Read PDF Content by extracting text using Python libraries such as pypdf.
- Apply basic promp engineering techniques to structure questions for models based on input text.
- Build a user interface with Gradio to interact with your document Q&A ssytem and switch between models.
- Learn the difference between pipeline(), AutoTokenizer, & AutoModelForCausalLM.

## What is Hugging Face
- Hugging Face has become a central hub for the Machine Learning community, especially for LLMs & Natural Language Processing (NLP).
- On Hugging Face Hub, You can find:
    - **Models**: Thousands of pre-trained models for various tasks (text generation, translation, image classification, etc.)
    - **Datasets**: A vast collection of datasets used to train and evaluated models.
    - **Spaces**: Demos of AI models hosted on HF infrastructure (like Gradio Apps.)

### Why Use Open-Source Models from Hugging Face.
- **Control**: You run the model yourself, giving you more control over data privacy and customization.
- **Cost**: Running smaller models can be cheaper than constantly hitting paid APIs, especially during development.
- **Transparency**: You can often study the model architecture and sometimes even the training data.
- **Community**: Access to a huge variety of models fine-tuned for specific tasks.
- **Offline Use**: Once downloaded, models can potentially be run without an internet connection.

## Install required Libraries, Obtain HF-Tokens and GPU Check.  
Install necessary Python libraries. We'll also need to potentially log in to Hugging Face if we want to use certain models (like some versions of llama or Gemma) that require user agreement.  
**Installing Libraries:**  
- `transformers`: The Core Hugging Face Library for models and tokenizer.
- `accelerate`: Helps run models efficiently across different hardware (like GPUs) and use less memory.
- `bitsandbytes`: Enables model quantization (like loading in 4-bit or 8-bit), drastically reducing memory usage. Essential for running decent models on free Colab GPUs!
- `torch`: The underlying deep learning framework (PyTorch)
- `pypdf`: A library to easily extract text from PDF Files.


In [13]:
# 1. CRITICAL: Upgrade transformers and related libraries
# After running this cell, you MUST restart the session to clear the cache.
import os

print("Upgrading transformers and installing libraries...")
!pip install -q --upgrade transformers accelerate bitsandbytes torch pypdf gradio

import transformers
print(f"Current Transformers Version: {transformers.__version__}")

if transformers.__version__ < '4.46.0':
    print("\n[!] VERSION TOO OLD: Please go to 'Runtime' -> 'Restart session' and run this cell again.")
else:
    print("\n[SUCCESS] Transformers is up to date. You can proceed with the rest of the notebook.")

Upgrading transformers and installing libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.1 MB/s eta 0:00

In [2]:
# Import Libraries
import torch    # PyTorch, the backend of the transformers
import pypdf    # For reading PDFs
import gradio as gr     # For Building the UserInterfaces.
from IPython.display import display, Markdown   # For nicer printing in notebook.
print("Core Libraries imported.")

Core Libraries imported.


### Hugging Face Hub Login:  
Some models on the Hugging Face Hub are "gated," meaning you need to agree to their terms and conditions before downloading. Logging in allows the `transformers` library to download these models if needed.

*   **Get a Hugging Face Token:**
    1.  Go to [huggingface.co](https://huggingface.co/).
    2.  Sign up or log in.
    3.  Click your profile picture (top right) -> Settings -> Access Tokens.
    4.  Create a new token (a 'read' role is usually sufficient).
    5.  Copy the generated token. **Treat this like a password!**
*   **Log in within Colab:** We'll use a helper function from the `huggingface_hub` library.

In [3]:
%pip install ipywidgets
# Install this, if error in the next cell

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 25.1 MB/s eta 0:00:00


In [4]:
import os
from huggingface_hub import login, notebook_login
print("Attempting Hugging Face login...")
# use notebook_login() for an interactive prompt in colab/notebook

notebook_login()
print("Login successful (or Token already Present)")

Attempting Hugging Face login...
Login successful (or Token already Present)


In [5]:
# Check if GPU is available (essential for running these models)
# Why GPU is Important: LLMs involve billions of calculations (matrix multiplocation)
# GPU are designed for massive parallel processing, making these calculations thousands of times faster than a standard CPU
# Running these models on a CPU would take an impractically long time (hours for a single answer instead of seconds/minutes)
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
    # set default device to GPU
    torch.set_default_device("cuda")
    print("PyTorch Default device set to CUDA (GPU).")

else:
    print("WARNING: No GPU detected. Running these models on CPU will be extremly slow")
    print("Make Sure 'GPU' is selected in Runtime > Change Run time Type (Google Colab)")

GPU Detected: Tesla T4
PyTorch Default device set to CUDA (GPU).


In [6]:
# Helper function for Markdown Display
def print_markdown(text):
    display(Markdown(text))

## Hugging Face Transformer Library: Pipelines

- `transformers`: A Python library that provides a standardized way to download, load, and use models from the Hub with jus a few lines of code.
    - `pipeline()`: A high-level, easy to use abstraction for common tasks (like text generation, summarization). Greate for quick tests and beginners.
    -   `AutoTokenizer()`: Automatically downloads the correct `tokenizer` for a model. A tokenizer converts human-readable text into numerical IDs the model understands.
    -   `AutoModelForCausalLM`: Automatically downloads the correct model architecture and pre-trained weights (e.g., `AutoModelForCausalLM` for text generation models like GPT, Llama, Gemma).
- `other Libraries`: HF also develops libraries like `accelerate` (for efficient loading/distributed training), `datasets` (for handling datasets), and `evaluate` (for model evaluation metrics).

#### What is a Transformer  
The Transformer architecture is an AI model for processing text, designed to understand and generate language efficiently.  
1. **Input Tokens (Words or Pieces of Words)**: It starts by breaking text into smaller chunks (tokens) like wrods or parts of words, for example, "I love cats" becomves [I, love, cats].
2. **Self Attention (Focus Mechanism)**: Imagine reading a sentence like: "The cat sat on the mat because it was soft." To Figure out what "it" refers to, you need to think about "mat". The Transformers uses self-attention to decide which words in the sequence are important to focus on. This helps it understand the context better.  
    Read the Paper at [Attention is All You Need](https://arxiv.org/abs/1706.03762).

3. **Layers (Like Thinking Steps)**: The model processes the input in multiple layers, where each layer refines its understanding. For example:
    - **Layer 1** might focus on word meanings.
    - **Layer 2** migh figure out relationships between words.
    - **Layer 3** might understand the whole sentence.
    - **Layer n** might work on xyz task.
4. **Positional Information (Word Order)**: Since the model reads all the words at once, it also adds positional encoding to know the order of the words (e.g., knowing "the cat" is different from "cat the").
5. **Output (Final Prediction)**: After preocessing, it predicts the next word, translates text, summarizes, or performs another task, for example, if you type "I love" it might predicts "cats".


In [7]:
# The popleines are a great and easy way to use models for inference.
# These pipelines are objects taht abstract most of the complex code from the library, offering a simpole API dedicated to several tasks
# Those tasks include Named Entity Recognition, Masked Language Modeling, Sentiment Analysis, Feature Extraction and Question Answering.
from transformers import pipeline

# Load a sentiment classifier model in financial news data
# Check the model here : https://huggingface.co/ProsusAI/finbert
pipe = pipeline(model = "ProsusAI/finbert")
pipe("Apple lost 10 Million dollars today due to US Tarrifs")

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'label': 'negative', 'score': 0.9706032276153564}]

### Hugging Face Transformer Library: Tokenizers

In [8]:
# Lets explore AutoTokenizer
# A Tokenizer converts text into numerical IDs that the model understands
# Check a demo for OpenAI's Tokenizers here: https://platform.openai.com/tokenizer
from transformers import AutoTokenizer

# Load tokenizer for GPT-2
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Encode text to token IDs
tokens = tokenizer("Hello everyone and Welcome to LLM and AI Agents Bootcamp")
print(tokens['input_ids'])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[15496, 2506, 290, 19134, 284, 27140, 44, 290, 9552, 28295, 18892, 16544]


### Hugging Face Transformers Library: AutoModelForCausalLM  
AutoModelForCausalLM is a Hugging Face class that automatically loads a pretrained model for causal (left-to-right) language modeling, such as GPT, LLaMA, or Gemma.  
Let's get hands-on and load a model! We'll start with a relatively small but capable model that should fit comfortably in Colab's free tier GPU memory, thanks to quantization.  
**Key Steps**:
1. **Choose a Model ID**: We need the unique identifier from the Hugging Face Hub (e.g. `"google/gemma-2b-it"` or `"microsoft/Phi-3-mini-4k-instruct"`).
2. **Load the Tokenizer** Use `AutoTokenizer.from_pretrained(model_id)` to get the specific tokenizer for that model.
3. **Load the Model**: Use `AutoModelForCausalLM.from_pretrained(...)` with crucial arguments:
  - `model_id`: The identifier.
  - `torch_dtype=torch.float16` (or `bfloat16`): Loads the model using 16-bit floating point numbers instead of 32-bit, saving memory.
  - `load_in_4bit=True` or `load_in_8bit=True`: This is **quantization** via `bitsandbytes`. It further reduces memory by representing model weights with fewer bits (4 or 8 instead of 16/32). Essential for free Colab! 4-bit saves more memory but might have a tiny impact on quality compared to 8-bit.
  - `device_map="auto"`: Tells `accelerate` to automatically figure out how to spread the model across available devices (primarily the GPU in our case).  
4. **Combine Tokenizer and Model (Optional but common)**: Using the `pipeline` fuunction is often simpler for basic text generation. It handles tokenization, model inference, and decoding back to text for you.

In [9]:
!pip install -U bitsandbytes

In [10]:
# CHoose a small, powerful model suitable for Colab.
# Alternatives, try (might need login/agreement)
# model_id = 'unsloth/gemma-3-4b-it-GGUP'
# model_id = "Qwen/Qwen2.5-B-Instruct"

model_id = "microsoft/Phi-4-mini-instruct"
# model_id = "unsloth/Llama-3.2-3B-Instruct"

In [11]:
# Load the Tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code = True)
print("Tokenizer loaded successfully.")

config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

configuration_phi3.py:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

Tokenizer loaded successfully.


In [12]:
# 1. UPGRADE & RESTART REQUIRED
# Run this cell, then go to: Runtime > Restart runtime
# After restarting, run this cell again.

import torch
try:
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer
except ImportError:
    !pip install -U transformers accelerate bitsandbytes
    print("RESTART RUNTIME NOW via 'Runtime > Restart runtime' to apply changes!")

# Create BitsAndBytesConfig for 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_compute_dtype = torch.float16,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_use_double_quant = True
)

print(f"Loading model: {model_id}")

# Load the model with the correct environment
model = AutoModelForCausalLM.from_pretrained(model_id,
                                             quantization_config = quantization_config,
                                             device_map = "auto",
                                             trust_remote_code = True)

Loading model: microsoft/Phi-4-mini-instruct


modeling_phi3.py:   0%|          | 0.00/54.3k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


ImportError: cannot import name 'LossKwargs' from 'transformers.utils' (/usr/local/lib/python3.12/dist-packages/transformers/utils/__init__.py)

In [ ]:
# Define a prompt
prompt = "Explain how Electric Vehicles work in a funny way!"


In [ ]:
prompt =  "What is the capital of Pakistan?""

In [ ]:
# Method # 1: Test the model and Tokenizer using the .generate() method

# Encode the input first
inputs = tokenizer(prompt, return_tensor = "pt")

# Generate the output
outputs = model.generate(**inputs, max_new_tokens= 1000)

response = tokenizer.decode(outputs[0], skip_special_tokens = True)

print_markdown(response)

In [ ]:
# Method 2: Alternatively, create a pipeline that includes model and tokenizer
# The pipeline wraps tokenization, generation, and decoding

pipe = pipeline("text_generation",
                model = model,
                tokenizer = tokenizer,
                torch_dtype = 'auto',   # Match model dtype
                device_map = 'auto'     # Ensure that pipeline uses the same device mapping
                )
outputs = pipe(prompt,
               max_new_tokens = 1000,   # max_new_tokens limits the length of the generated response
               temperature = 1       # Temperature controls randomness (lower = more focused)

)


# print the generated text
print_markdown(outputs[0]['generated_text'])

### Read PDF Documents and Extract Text using PyPDF Library  

Now that the model is loaded, we need text from our documents to ask questions about. We'll use the pypdf library to extract text from a PDF file.  
For this example, we'll download a sample PDF about qurterly earning report. You can easily adapt this to use your own PDF by uploading it to Colab.

**Steps**:
1. **Get the PDF**: Download it or specify the path if uploaded.
2. **Open the PDF**: Use `pypdf.PdfReader`.
3. **Iterate Through Pages**: Loop through each page in the PDF.
4. **Extract Text**: Use `page.extract_text()`.
5. **Combine Text**: Join the text from all pages into a single string.

In [ ]:
import requests
from pathlib import Path

# ---- Get the PDF File from the follwoing link ----
pdf_url = "https://abc.xyz/assets/66/ae/c94682fc4137b5fb90a5d709ac4b/2025-q1-earnings-transcript.pdf"

pdf_filename = 'google_earning_transcript.pdf'
pdf_path = Path(pdf_filename)

# Download the file if it doesn't exist
if not pdf_path.exists():
  response =requests.get(pdf_url)
  response.raise_for_status()   # Check for download errors
  pdf_path.write_bytes(response.content)
  print(f"PDF Downloaded successfully to {pdf_path}")
else:
  print(f"PDF File already exists at {pdf_path}")

# ---- read Text from PDF using pypdf ---
pdf_text = ""

print(f"Reading text from {pdf_path}....")
reader = pypdf.PdfReader(pdf_path)
num_pages = len(reader.pages)
print(f"PDF has {num_pages} pages.")

# Extract text from each page

all_pages_text = []

for i, page in enumerate(reader.pages):
  page_text = page.extract_text()
  if page_text:   # Only add if text extraction was successful
    all_pages_text.append(page_text)
  print(f"Read page {i+1}/{num_pages}")

# Join the text from all pages
pdf_text = "\n".join(all_pages_text)
print(f"Successfully extracted text. Total Characters: {len(pdf_text)}")

In [ ]:
# Dsplay a snippet of the PDF
print("\n--- Snippet of Extracted Text ---")
print_markdown(f"{pdf_text[:500]}")

### Build the Q&A Logic and Prompt the Model  
Now we have the two key ingredients  
1. A loaded open-Source LLM (and its tokenizer/pipeline)
2. The text content extracted from out PDF document  

We need to combine these to answer user questions. The core idea is **prompt engineering**: We'll crete a prompt that includes both the user's question and the relevant document as a context, instructing the model to answer based only on that context.  
**Steps**:
1. **Define a Prompt Tempate**: Create a string that structures the input for the LLM. This typically includes placeholders for the context (PDF text) and the question.
2. **Create an Answering Function**: Write a Python function that takes the PDF text, the user question, and the model/tokenizer (or pipeline) as input.
3. **Format the Prompt**: Inside the function, fill the template with the actual PDF text and question.
4. **Handle Context Length**: LLMs have a maximum context window (how much text they can read at once). Our sample PDF might be too long! For simplicity now, we might just truncate the PDF text if it's excessive. More advanced techniques involve chunking the document and retrieving only relevant parts, but we'll keep it basic here.
5. **Run Inference**: Send the formatted prompt to the model pipeline.
6. **Extract the Answer**: Process the model's output to get just the answer part.

In [2]:
MAX_CONTEXT_CHARS = 6000
def answer_question_from_pdf(document_text, question, llm_pipeline):
  if len(document_text) > MAX_CONTEXT_CHARS:
    print(f'Warning: Truncating context...')
    context = document_text[:MAX_CONTEXT_CHARS] + '...'
  else:
    context = document_text

  prompt_template = f"""<|system|>
  Answer based only on the provided text.

  Document Text:
  ---
  {context}
  ---
  <|end|>
  <|user|>
  Question : {question}
  <|end|>
  <|assistant|>
  Answer: """

  outputs = llm_pipeline(
      prompt_template,
      max_new_tokens = 500,
      do_sample = True,
      temperature = 0.2,
      top_p = 0.9
  )

  full_generated_text = outputs[0]['generated_text']
  if "Answer:" in full_generated_text:
    answer_start_index = full_generated_text.find("Answer:") + len("Answer:")
    raw_answer = full_generated_text[answer_start_index:].strip()
  else:
    raw_answer = full_generated_text

  end_token = "<|end|>"
  if end_token in raw_answer:
    raw_answer = raw_answer.split(end_token)[0]

  return raw_answer

In [ ]:
# Test the function
test_question = "What is this document about>"
geenrated_answer  =answer_question_from_pdf(pdf_text, test_question, pipe)

print("\nTest Question: ")
print_markdown(f"**Q:** {test_question}")
print("\nGenerated Answer: ")
print_markdown(f"**A:** {generated_answer}")

### Switch Models and Buidl Gradio Interface

We have a working Q&A system with one model. But the Beauty of Hugging Face is the vast choice! Let's adapt our setup to easily switch between different open-source models and build a Gradio interface to interact with it.  
**Challenge & Approach**:
  - **Loading Multiple Models**: Loading several LLMs simultaneously (even quantized) will liekly exceed Colab's GPU memory.
  - **Solution**: We'll load one model at a time based on the user's selection in the Gradio interface. This means unloading the previous model before loading the new one. This will introduce a loading delay when switching models, but it's necessary for memory constraints.
**Steps**:
  1. **Define Model Choices**: Create a dictionary mapping user-friendly names (e.g. "phi-3 Mini") to their Hugging Face model IDs. Include models known to work in Colab free tier with 4-bit quantization.
  2. **Global State**: Keep track of the currently loaded model and tokenizer globally (or using Gradio's `State`).
  3. **Model Loading Function**: Create a function `load_model(model_id)` that handles unloading the old model (if any) and laoding the new tokenizer and quantized model. It should return the new `pipeline`.
  4. **Gradio Interface**:
    - Use `gr.Blocks` for more layout control.
    - Add a `gr.Dropdown` for the user to select the desired model.
    - Add a `gr.Textbox` for the user's question.
    - Add a `gr.Textbox` (or `gr.markdown`) for the output answer.
    - Add a `gr.Button`) to submit the question.
  5. **Event Handling**:
    - When the dropdown selection changes, trigger the `load_model` function. Show a loading indicator.
    - When the submit button is clicked, call `answer_question_from_pdf` function, passing the current PDF text, the question, and the currently loaded pipeline.


In [ ]:
# make sure we have the pdf_text
# Configuration : Model available for selection.
# Use models knwon to fit in Colab free tier with 4-bit quantization.

available_models = {
    "Llama 3.2" : "unsloth/Llama-3.2-3B-Instruct",
    "Microsoft Phi-4 Mini" : "microsoft/Phi-4-mini-instruct",
    "Google Gemma 3" : "unsloth/gemma-3-4b-it-GGUP"
}

In [3]:
# --- Global State ---
available_models = {
    "Llama 3.2" : "unsloth/Llama-3.2-3B-Instruct",
    "Microsoft Phi-4 Mini" : "microsoft/Phi-4-mini-instruct",
    "Google Gemma 3" : "unsloth/gemma-3-4b-it-GGUP"
}

current_model_id = None
current_pipeline = None
print(f'Models available for selection : {list(available_models.keys())}')

def load_llm_model(model_name):
  global current_model_id, current_pipeline, tokenizer, model
  new_model_id = available_models.get(model_name)
  if not new_model_id: return "Invalid model selected.", None

  if new_model_id == current_model_id and current_pipeline is not None:
    return f"{model_name} already loaded", current_pipeline

  print(f"Switching to model: {model_name}...")
  current_pipeline = None
  import gc
  if "model" in globals(): del globals()["model"]
  if "tokenizer" in globals(): del globals()["tokenizer"]
  torch.cuda.empty_cache()
  gc.collect()

  try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
    tokenizer = AutoTokenizer.from_pretrained(new_model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(new_model_id, torch_dtype='auto', load_in_4bit=True, device_map='auto', trust_remote_code=True)
    current_pipeline = pipeline('text-generation', model=model, tokenizer=tokenizer)
    current_model_id = new_model_id
    return f"{model_name} loaded successfully!", current_pipeline
  except Exception as e:
    return f"Error Loading {model_name}: {e}", None

Models available for selection : ['Llama 3.2', 'Microsoft Phi-4 Mini', 'Google Gemma 3']


In [5]:
# --- Functio to Hanlge Q&A Submission ---
# This functino now relies on the globally managed 'current_pipeline'
# In a more robust Gradio app, you'd pass the pipeline via gr.State
def handle_submit(question):
  """
  Handles the user submitting a question.
  """
  if not current_peipeline:
    return "Error: No model is currently loaded. Please select a model"
  if not pdf_text:
    return "Error: PDF Text is not loaded. Please run Section 4."
  if not question:
    return "Please enter a question"

  print(f'Hanlding submission for question: "{question}" using {current_model}')
  # Call the Q&A function defined in Section 5
  answer = answer_question_from_pdf(pdf_text, question, current_pipeline)
  return answer


In [7]:
# --- Build Gradio Interface ---
import gradio as gr
print("Building Gradio Interface...")
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# PDF Q&A Bot\nAsk questions about the document. Select an LLM to answer.")

    with gr.Row():
        with gr.Column():
            model_dropdown = gr.Dropdown(choices=list(available_models.keys()), label="Select LLM Model", value=list(available_models.keys())[0])
            status_textbox = gr.Textbox(label="Model Status", interactive=False)
            question_textbox = gr.Textbox(label="❓ Your Question", lines=2)
            submit_button = gr.Button("Submit Question", variant='primary')
        with gr.Column():
            answer_textbox = gr.Textbox(label="💡 Answer", lines=10, interactive=False)

    model_dropdown.change(fn=load_llm_model, inputs=[model_dropdown], outputs=[status_textbox])
    submit_button.click(fn=lambda q: answer_question_from_pdf(pdf_text, q, current_pipeline) if current_pipeline else "Load a model first", inputs=[question_textbox], outputs=[answer_textbox])

print("Launching Gradio demo...")
demo.launch(debug=True)

Building Gradio Interface...


/tmp/ipykernel_4326/1511265870.py:4: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Launching Gradio demo...
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a6bb9bc5f734b48b3c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dis

Switching to model: Microsoft Phi-4 Mini...


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 386, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2280, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1657, in call_function
    prediction = await anyio.to_thread.run_sync(  # type:

Switching to model: Google Gemma 3...


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 386, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2280, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1657, in call_function
    prediction = await anyio.to_thread.run_sync(  # type:

Switching to model: Llama 3.2...
Keyboard interruption in main thread... closing server.


KeyboardInterrupt: 